# 03 — Popularidade e baselines

**EP · Data Science Knowledge Base | Sistemas de Recomendação | Notebook 03/16**

**Execução:** Reinicie o kernel e execute as células em ordem. Observe as instruções específicas e as dependências opcionais indicadas neste notebook.


## Materiais complementares

- [Guia Técnico](reference_materials/03_popularidade_baselines.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_03_popularidade_baselines.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
- Criar baseline de média e contagem
- Usar suavização para reduzir viés de amostras pequenas

## Fundamentação conceitual
A média simples pode favorecer itens com uma única nota. Uma alternativa é a média bayesiana $s_i=(n_i\bar r_i+mC)/(n_i+m)$, com média global $C$ e força de regularização $m$.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## Ranking simples

In [2]:
stats=ratings.groupby('movieId').rating.agg(['mean','count']).reset_index()
print(stats.sort_values(['mean','count'],ascending=False).head(5).merge(movies,on='movieId')[['title','mean','count']].to_string(index=False))

                                                                            title  mean  count
                                                                  Lamerica (1994)   5.0      2
                                             Heidi Fleiss: Hollywood Madam (1995)   5.0      2
                                                              Lesson Faust (1994)   5.0      2
Jonah Who Will Be 25 in the Year 2000 (Jonas qui aura 25 ans en l'an 2000) (1976)   5.0      2
                                                              Belle époque (1992)   5.0      2


**Interpretação.** Observe quantos votos sustentam cada média.

## Ranking suavizado

In [3]:
C=ratings.rating.mean(); m=30
stats['score']=(stats['count']*stats['mean']+m*C)/(stats['count']+m)
rank=stats.merge(movies,on='movieId').sort_values('score',ascending=False)
print(rank[['title','mean','count','score']].head(10).to_string(index=False))

                                                title     mean  count    score
                     Shawshank Redemption, The (1994) 4.429022    317 4.348838
                                Godfather, The (1972) 4.289062    192 4.182643
                                    Fight Club (1999) 4.272936    218 4.179624
            Star Wars: Episode IV - A New Hope (1977) 4.231076    251 4.153191
                           Usual Suspects, The (1995) 4.237745    204 4.143362
                              Schindler's List (1993) 4.225000    220 4.138187
                                  Pulp Fiction (1994) 4.197068    307 4.135153
Star Wars: Episode V - The Empire Strikes Back (1980) 4.215640    211 4.126750
                                   Matrix, The (1999) 4.192446    278 4.125152
                       Godfather: Part II, The (1974) 4.259690    129 4.116646


**Interpretação.** A suavização reduz a confiança em médias de poucos registros.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** popularidade suavizada. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **popularidade suavizada** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Laboratório adicional — Média suavizada bayesiana

Experimento independente do estado das células anteriores. Use-o como verificação e explique seus limites; resultados em conjuntos sintéticos não são estimativas de desempenho real.

In [4]:
import numpy as np
notas=np.array([5,5],dtype=float);media_global=3.5;forca=5
score=(notas.sum()+forca*media_global)/(len(notas)+forca)
print('Média bruta:',notas.mean(),'média suavizada:',round(score,3))

Média bruta: 5.0 média suavizada: 3.929


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **popularidade suavizada** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — Suavização e suporte de avaliações

Para um item `i`, a média suavizada é `s_i = (n_i * r̄_i + m*C)/(n_i+m)`, com `C` igual à média global e `m` representando a força de suavização. Essa expressão pode ser vista como um **estimador com prior empírico**, e não como um posterior bayesiano universal. Ajuste `m` apenas em treino/validação.

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [5]:
# Demonstração independente do conjunto de dados, para conferir limites
media_global = 3.5
for n, media in [(1, 5.0), (20, 5.0)]:
    s = (n*media + 30*media_global)/(n+30)
    print(f'n={n}: média observada={media:.1f}, suavizada={s:.3f}')
    assert media_global <= s <= media

n=1: média observada=5.0, suavizada=3.548
n=20: média observada=5.0, suavizada=4.100


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

## Limitações e boas práticas
O ranking de popularidade ainda é não personalizado e sujeito ao efeito de exposição.

## Exercícios
Experimente `m=5` e `m=100`. Que filmes mudam de posição?

## Soluções comentadas
Quanto maior `m`, mais os itens com poucas avaliações são puxados para a média global.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados do capítulo em `outputs/`.


In [6]:
# EP_OUTPUTS_09_V1
# Exportação após os experimentos. Executar este notebook desde o início (Run All).
# Arquivos com prefixo do capítulo evitam colisões com outras saídas.
from pathlib import Path
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
rank[["movieId", "title", "mean", "count", "score"]].rename(columns={
    "mean": "nota_media", "count": "n_avaliacoes", "score": "media_suavizada"
}).to_csv(out / "03_ranking_popularidade_suavizada.csv", index=False)
print('Outputs do capítulo 03 salvos em:', out.resolve())


Outputs do capítulo 03 salvos em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/09_recommender_systems/outputs
